# Visualization of FAIRFluids Data
 
 This notebook loads and visualizes multiple FAIRFluids JSON documents.
 
 **Features:**
 - Loads multiple FAIRFluids JSON files from a directory
 - Visualizes viscosity over time for all fluids
 - Groups by composition (Molar Fractions)
 - Different markers for storage conditions
 - Supports multiple compound combinations simultaneously

In [23]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict
from fairfluids.core.lib import FAIRFluidsDocument

plt.rcParams['font.size'] = 10

In [ ]:
# Load multiple JSON documents from a directory
output_dir = Path('outputs_des_storage')

# Option 1: Load all JSON files from the directory
#json_files = list(output_dir.glob('*.json'))

# Option 2: Or select specific files
json_files = [
    output_dir / 'Betaine_Ethylene_glycol_Water.json',
    output_dir / 'Betaine_Glycerol_Water.json',
    output_dir / 'Cholinchloride_Ethylene_glycol_Water.json',
    output_dir / 'Cholinchloride_Glycerol_Water.json',
    output_dir / 'Lidocaine_Oxalic_acid_Water.json',
    output_dir / 'Tetrabutylammonium_c_Glycerol_Water.json',

]

print(f"Found JSON files: {len(json_files)}")

# Load all documents
documents = []
for json_path in json_files:
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            doc = FAIRFluidsDocument.model_validate_json(f.read())
            documents.append(doc)
            print(f"  ✓ {json_path.name}: {len(doc.fluid)} Fluids, Compounds: {[c.compoundID for c in doc.compound]}")
    except Exception as e:
        print(f"  ✗ Error loading {json_path.name}: {e}")

print(f"\nTotal: {len(documents)} documents loaded")
total_fluids = sum(len(doc.fluid) for doc in documents)
print(f"Total: {total_fluids} Fluids across all documents")

Found JSON files: 2
  ✓ Betaine_Ethylene_glycol_Water.json: 12 Fluids, Compounds: ['Betaine', 'Ethylene Glycol', 'Water']
  ✓ Betaine_Glycerol_Water.json: 12 Fluids, Compounds: ['Betaine', 'Glycerol', 'Water']

Total: 2 documents loaded
Total: 24 Fluids across all documents


### Function definitions

In [12]:
# Visualize property over parameter for all documents
# Each composition gets its own color
# Each storage condition gets its own marker

def plot_property_over_parameter(data_by_composition, 
                                  parameter_label="Time (days)", 
                                  property_label="Viscosity (mPa·s)",
                                  title="Viscosity over Time",
                                  num_documents=None):
    """
    Visualize property data over a parameter for all compositions.
    
    Args:
        data_by_composition: Dictionary with data grouped by composition and storage condition
        parameter_label: Label for x-axis (e.g. "Time (days)", "Temperature (K)")
        property_label: Label for y-axis (e.g. "Viscosity (mPa·s)", "Density (kg/m³)")
        title: Plot title
        num_documents: Number of documents (optional, for title)
    """
    fig, ax = plt.subplots(figsize=(14, 10))

    # Markers for different storage conditions
    markers = {
        'None': 'o',
        'Closed': 's',
        'Fridge': '^',
        'Open': 'D'
    }

    # Use a larger colormap for many compositions
    num_compositions = len(data_by_composition)
    if num_compositions <= 10:
        colors = plt.cm.tab10(np.linspace(0, 1, num_compositions))
    elif num_compositions <= 20:
        colors = plt.cm.tab20(np.linspace(0, 1, num_compositions))
    else:
        # For even more compositions, use a continuous colormap
        colors = plt.cm.viridis(np.linspace(0, 1, num_compositions))

    # Count the number of plotted series for the legend
    plotted_series = 0
    max_legend_entries = 30  # Limit the number of legend entries

    for comp_idx, (comp_key, storage_data) in enumerate(data_by_composition.items()):
        color = colors[comp_idx]
        
        # Create label for this composition
        comp_dict = dict(comp_key)
        comp_label = ", ".join([f"{k}: {v:.3f}" for k, v in comp_dict.items()])
        
        for storage_condition, measurements in storage_data.items():
            if not measurements:
                continue
            
            # Sort by parameter value
            sorted_meas = sorted(measurements, key=lambda x: x['parameter'])
            parameters = [m['parameter'] for m in sorted_meas]
            properties = [m['property'] for m in sorted_meas]
            uncertainties = [m['uncertainty'] if m['uncertainty'] is not None else 0 for m in sorted_meas]
            
            # Label for each combination of composition and storage condition
            label = f"{comp_label} ({storage_condition})"
            
            # Show label only if we still have space in the legend
            show_label = plotted_series < max_legend_entries
            
            # Plot with error bars
            ax.errorbar(
                parameters,
                properties,
                yerr=uncertainties,
                marker=markers.get(storage_condition, 'o'),
                color=color,
                label=label if show_label else None,
                linestyle='-',
                linewidth=1.5,
                markersize=8,
                capsize=3,
                alpha=0.7
            )
            
            if show_label:
                plotted_series += 1

    ax.set_xlabel(parameter_label, fontsize=12)
    ax.set_ylabel(property_label, fontsize=12)
    
    # Create title with optional document count
    full_title = title
    if num_documents is not None:
        full_title += f' - All Documents ({num_documents} documents, {num_compositions} compositions)'
    else:
        full_title += f' ({num_compositions} compositions)'
    
    ax.set_title(full_title, fontsize=14, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=1)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=-1)

    if plotted_series >= max_legend_entries:
        ax.text(0.02, 0.98, f'Note: Only first {max_legend_entries} series in legend', 
                transform=ax.transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()

def extract_property_data(documents, property_id="viscosity", parameter_id="parameter_time", show_summary=True):
    """
    Extract property data from all documents.
    Group by composition (molar fractions) and storage condition.
    
    Args:
        documents: List of FAIRFluidsDocument objects
        property_id: ID of the property to extract (e.g. "viscosity", "density")
        parameter_id: ID of the parameter for the x-axis (e.g. "parameter_time", "parameter_temperature")
        show_summary: If True, show summary of extracted data
    
    Returns:
        dict: Nested dictionary structure with data grouped by composition and storage condition
    """
    data_by_composition = defaultdict(lambda: defaultdict(list))
    
    # Iterate through all documents
    for doc_idx, doc in enumerate(documents):
        print(f"\nProcessing document {doc_idx + 1}/{len(documents)}: {len(doc.fluid)} fluids")
        
        for fluid in doc.fluid:
            # Extract molar fractions for this composition
            # All measurements in a fluid have the same composition
            molar_fractions = {}
            if fluid.measurement:
                # Take the first measurement as reference for the composition
                first_meas = fluid.measurement[0]
                for param in fluid.parameter:
                    # Check if it's a mole fraction parameter
                    param_type = None
                    if hasattr(param.parameter, 'value'):
                        param_type = param.parameter.value
                    elif isinstance(param.parameter, str):
                        param_type = param.parameter
                    
                    if param_type == "Mole fraction":
                        if param.associated_compounds:
                            comp_name = param.associated_compounds[0]
                            # Find the corresponding ParameterValue in the first measurement
                            for pv in first_meas.parameterValue:
                                if pv.parameterID == param.parameterID:
                                    molar_fractions[comp_name] = pv.paramValue
                                    break
            
            # Create a unique key for the composition (sorted for consistency)
            comp_key = tuple(sorted(molar_fractions.items()))
            
            # Extract storage condition
            storage_condition = None
            if fluid.storage and fluid.storage.storage_condition:
                if hasattr(fluid.storage.storage_condition, 'value'):
                    storage_condition = fluid.storage.storage_condition.value
                else:
                    storage_condition = str(fluid.storage.storage_condition)
            else:
                storage_condition = "None"
            
            # Extract property measurements
            for meas in fluid.measurement:
                # Check if the desired property is present
                property_value = None
                property_uncertainty = None
                
                for pv in meas.propertyValue:
                    if pv.propertyID == property_id:
                        property_value = pv.propValue
                        property_uncertainty = pv.uncertainty
                        break
                
                # If property not found, skip this measurement
                if property_value is None:
                    continue
                
                # Find parameter value (e.g. time or temperature)
                parameter_value = None
                for param_val in meas.parameterValue:
                    if param_val.parameterID == parameter_id:
                        parameter_value = param_val.paramValue
                        break
                
                # If parameter not found, skip this measurement
                if parameter_value is None:
                    continue
                
                # Also add compound information for better distinction
                compounds_str = ", ".join(fluid.compounds) if hasattr(fluid, 'compounds') and fluid.compounds else "Unknown"
                
                data_by_composition[comp_key][storage_condition].append({
                    'parameter': parameter_value,
                    'property': property_value,
                    'uncertainty': property_uncertainty,
                    'compounds': compounds_str,
                    'doc_idx': doc_idx
                })
    
    # Show summary if requested
    if show_summary:
        print(f"\n{'='*60}")
        print(f"Summary across all documents:")
        print(f"{'='*60}")
        print(f"Found unique compositions: {len(data_by_composition)}")
        
        total_measurements = 0
        for comp_key, storage_data in data_by_composition.items():
            comp_measurements = sum(len(meas) for meas in storage_data.values())
            total_measurements += comp_measurements
            print(f"\nComposition: {dict(comp_key)}")
            print(f"  Total: {comp_measurements} measurements")
            for storage, measurements in storage_data.items():
                if measurements:
                    print(f"    {storage}: {len(measurements)} measurements")
        
        print(f"\n{'='*60}")
        print(f"Total: {total_measurements} {property_id.capitalize()} measurements")
        print(f"{'='*60}")
    
    return data_by_composition


In [18]:
# Visualize property over parameter for all documents using Plotly
# Each composition gets its own color
# Each storage condition gets its own marker

def plotly_property_over_parameter(data_by_composition, 
                                  parameter_label="Time (days)", 
                                  property_label="Viscosity (mPa·s)",
                                  title="Viscosity over Time",
                                  num_documents=None):
    """
    Visualize property data over a parameter for all compositions using Plotly.
    
    Args:
        data_by_composition: Dictionary with data grouped by composition and storage condition
        parameter_label: Label for x-axis (e.g. "Time (days)", "Temperature (K)")
        property_label: Label for y-axis (e.g. "Viscosity (mPa·s)", "Density (kg/m³)")
        title: Plot title
        num_documents: Number of documents (optional, for title)
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Markers for different storage conditions
    markers = {
        'None': 'circle',
        'Closed': 'square',
        'Fridge': 'triangle-up',
        'Open': 'diamond'
    }

    # Use a color palette for compositions
    num_compositions = len(data_by_composition)
    if num_compositions <= 10:
        import plotly.express as px
        colors = px.colors.qualitative.Plotly
    elif num_compositions <= 20:
        import plotly.express as px
        colors = px.colors.qualitative.Light24
    else:
        # For even more compositions, use a continuous colormap
        import plotly.express as px
        colors = px.colors.sample_colorscale("viridis", [i/(num_compositions-1) for i in range(num_compositions)])

    fig = go.Figure()

    for comp_idx, (comp_key, storage_data) in enumerate(data_by_composition.items()):
        color = colors[comp_idx % len(colors)]
        
        # Create label for this composition
        comp_dict = dict(comp_key)
        comp_label = ", ".join([f"{k}: {v:.3f}" for k, v in comp_dict.items()])
        
        for storage_condition, measurements in storage_data.items():
            if not measurements:
                continue
            
            # Sort by parameter value
            sorted_meas = sorted(measurements, key=lambda x: x['parameter'])
            parameters = [m['parameter'] for m in sorted_meas]
            properties = [m['property'] for m in sorted_meas]
            uncertainties = [m['uncertainty'] if m['uncertainty'] is not None else 0 for m in sorted_meas]
            
            # Label for each combination of composition and storage condition
            label = f"{comp_label} ({storage_condition})"
            
            # Add trace with error bars
            fig.add_trace(go.Scatter(
                x=parameters,
                y=properties,
                error_y=dict(
                    type='data',
                    array=uncertainties,
                    visible=True
                ),
                mode='lines+markers',
                marker=dict(
                    symbol=markers.get(storage_condition, 'circle'),
                    size=8,
                    color=color,
                    line=dict(width=1, color='white')
                ),
                line=dict(
                    color=color,
                    width=1.5
                ),
                name=label,
                showlegend=True,
                opacity=0.7
            ))

    # Create title with optional document count
    full_title = title
    if num_documents is not None:
        full_title += f' - All Documents ({num_documents} documents, {num_compositions} compositions)'
    else:
        full_title += f' ({num_compositions} compositions)'
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=full_title,
            font=dict(size=14, family='Arial, sans-serif')
        ),
        xaxis=dict(
            title=parameter_label,
            titlefont=dict(size=12),
            gridcolor='lightgray',
            gridwidth=0.5,
            range=[-1, None]
        ),
        yaxis=dict(
            title=property_label,
            titlefont=dict(size=12),
            gridcolor='lightgray',
            gridwidth=0.5
        ),
        legend=dict(
            font=dict(size=8),
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        ),
        width=1400,
        height=1000,
        hovermode='closest',
        plot_bgcolor='white'
    )

    fig.show()


In [19]:
# Extrahiere Viskositätsdaten aus allen Dokumenten
data_by_composition = extract_property_data(
    documents, 
    property_id="viscosity", 
    parameter_id="parameter_time",
    show_summary=False
)

data_by_composition
# Plotte Viskosität über Zeit
plotly_property_over_parameter(
    data_by_composition,
    parameter_label="Time (days)",
    property_label="Viscosity (mPa·s)",
    title="Viscosity over Time",
    num_documents=len(documents)
)



Processing document 1/6: 12 fluids

Processing document 2/6: 12 fluids

Processing document 3/6: 12 fluids

Processing document 4/6: 12 fluids

Processing document 5/6: 12 fluids

Processing document 6/6: 12 fluids


## Legende

- **Farben**: Jede Farbe repräsentiert eine eindeutige Komposition (Molar Fractions) über alle Dokumente
- **Marker**:
  - `o` (Kreis): None (keine Storage Condition)
  - `s` (Quadrat): Closed
  - `^` (Dreieck): Fridge
  - `D` (Diamant): Open



In [21]:
# Extract polarity data from all documents
data_by_composition = extract_property_data(
    documents, 
    property_id="polarity", 
    parameter_id="parameter_time",
    show_summary=False
)

data_by_composition
# Plot polarity over time
plotly_property_over_parameter(
    data_by_composition,
    parameter_label="Time (days)",
    property_label="Polarity",
    title="Polarity over Time",
    num_documents=len(documents)
)


Processing document 1/6: 12 fluids

Processing document 2/6: 12 fluids

Processing document 3/6: 12 fluids

Processing document 4/6: 12 fluids

Processing document 5/6: 12 fluids

Processing document 6/6: 12 fluids
